# MNIST — progressive CNN design study

Three architectures of increasing depth and regularisation, trained under one
protocol. A thin driver over `src/mnist.py`.

| Model | Conv blocks | BatchNorm | Dropout |
|-------|-------------|-----------|---------|
| ShallowCNN | 1 | no | no |
| DualBlockCNN | 2 | yes | no |
| DeepRegularisedCNN | 3 | yes | 0.4 |

In [ ]:
# Run from anywhere inside the repo: put the project root on sys.path.
import sys
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "src").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

%load_ext autoreload
%autoreload 2

In [ ]:
from src.mnist import MODELS, build_loaders, count_params, plot_samples
from src.utils import get_device, set_seed

set_seed()
device = get_device()
loaders, train_full = build_loaders()

print("device:", device)
for split, loader in loaders.items():
    print(f"  {split:<15}{len(loader.dataset):>7,} images")

The official 10,000-image test set is kept as a genuine held-out set. The
60,000 training images are split 42k / 9k / 9k, so `inner_test` is a held-out
slice of the *training* distribution and `official_test` is the real benchmark.
Both are reported.

In [ ]:
print('saved', plot_samples(train_full))

## Train

In [ ]:
from src.mnist import train_model, evaluate

histories, summary, predictions = {}, [], {}
for name, factory in MODELS.items():
    set_seed()
    model, history = train_model(factory(), name, loaders, device)
    histories[name] = history

    inner_acc, _, _ = evaluate(model, loaders["inner_test"], device)
    official_acc, preds, labels = evaluate(model, loaders["official_test"], device)
    predictions[name] = {"preds": preds, "labels": labels}

    summary.append({"model": name, "params": count_params(model),
                    "best_val_acc": max(history["val_acc"]),
                    "inner_test_acc": inner_acc,
                    "official_test_acc": official_acc,
                    "epochs": len(history["train_loss"])})
    print(f"{name}: held-out {inner_acc:.4f} | official test {official_acc:.4f}")

## Results

In [ ]:
import pandas as pd

table = pd.DataFrame(summary)
table["best_val_acc"] = (table["best_val_acc"] * 100).round(2)
table["inner_test_acc"] = (table["inner_test_acc"] * 100).round(2)
table["official_test_acc"] = (table["official_test_acc"] * 100).round(2)
table

In [ ]:
from src.mnist import plot_confusion_matrices, plot_learning_curves

print("saved", plot_learning_curves(histories))
print("saved", plot_confusion_matrices(predictions))

In [ ]:
from sklearn.metrics import classification_report

for name, payload in predictions.items():
    print(f"\n{name} — official MNIST test set")
    print(classification_report(payload["labels"], payload["preds"], digits=4))